# Notebook 18 — Spectral Memory + Low-Rank Operator Correction

**prime-numbers-lab**

This notebook continues Notebook 17's higher-order transition-memory path. Notebook 17 showed that empirical two-step transition structure can deviate from a first-order Markov baseline. Notebook 18 asks whether that deviation is structured enough to be captured by a small number of spectral memory modes.

Notebook 18 keeps the locked repo template:

```text
18_spectral_memory_low_rank_operator/
  figures/
  data/
  docs/
  tex/
18_spectral_memory_low_rank_operator_export.zip
```

Core question:

> Can low-rank spectral corrections explain the empirical two-step transition residual beyond the Markov baseline?

Outputs include transition operators, singular-vector diagnostics, rank-k corrected operators, prediction-error curves, residue projections, CSV summaries, Markdown notes, and a compact LaTeX section.


In [ ]:
# ============================================================
# Notebook 18 locked-template setup
# ============================================================

import os
import math
import json
import zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "18_spectral_memory_low_rank_operator"
OUTDIR = Path(NOTEBOOK_ID)
FIGDIR = OUTDIR / "figures"
DATADIR = OUTDIR / "data"
DOCDIR = OUTDIR / "docs"
TEXDIR = OUTDIR / "tex"

for d in [OUTDIR, FIGDIR, DATADIR, DOCDIR, TEXDIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 9423
rng = np.random.default_rng(SEED)

X_MAX = 2_000_000
WINDOWS = 13
N_STATES = 8
TOP_RANK = N_STATES

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "font.size": 12,
})

print("Notebook:", NOTEBOOK_ID)
print("Output directory:", OUTDIR.resolve())

## 1. Prime data and normalized gap states

We compute primes up to `X_MAX`, prime gaps

\[
g_n = p_{n+1}-p_n,
\]

and normalized gaps

\[
z_n=\frac{g_n}{\log(p_n)}.
\]

As in Notebook 17, normalized gaps are discretized into quantile states. Notebook 18 also keeps prime anchor residues \(p_n \bmod 30\) so spectral modes can be projected back onto residue classes.

In [ ]:
# ============================================================
# Prime generator and normalized gap features
# ============================================================

def sieve_primes(n: int) -> np.ndarray:
    """Return all primes <= n using a bytearray sieve."""
    if n < 2:
        return np.array([], dtype=np.int64)
    sieve = bytearray(b"\x01") * (n + 1)
    sieve[0:2] = b"\x00\x00"
    limit = int(n**0.5)
    for p0 in range(2, limit + 1):
        if sieve[p0]:
            start = p0 * p0
            sieve[start:n+1:p0] = b"\x00" * (((n - start) // p0) + 1)
    return np.fromiter((i for i, v in enumerate(sieve) if v), dtype=np.int64)

primes = sieve_primes(X_MAX)
p = primes[:-1]
gaps = np.diff(primes).astype(float)
logp = np.log(p)
z = gaps / logp

mask = np.isfinite(z) & (p >= 101)
p = p[mask]
gaps = gaps[mask]
logp = logp[mask]
z = z[mask]
anchor_mod30 = (p % 30).astype(int)
valid_residues = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)

quantile_edges = np.quantile(z, np.linspace(0, 1, N_STATES + 1))
quantile_edges[0] = -np.inf
quantile_edges[-1] = np.inf
states = np.digitize(z, quantile_edges[1:-1], right=False).astype(int)

# Logarithmic windows.
edges = np.geomspace(max(101, int(p.min())), int(p.max()), WINDOWS + 1).astype(int)
window_index = np.full(len(p), -1, dtype=int)
window_rows = []
for i in range(WINDOWS):
    lo, hi = edges[i], edges[i + 1]
    if i == WINDOWS - 1:
        m = (p >= lo) & (p <= hi)
    else:
        m = (p >= lo) & (p < hi)
    window_index[m] = i
    if m.sum() > 0:
        window_rows.append({
            "window_index": i,
            "x_min": int(lo),
            "x_max": int(hi),
            "window_midpoint": float(np.sqrt(lo * hi)),
            "count": int(m.sum()),
            "mean_z": float(np.mean(z[m])),
            "std_z": float(np.std(z[m])),
            "q50_z": float(np.quantile(z[m], 0.50)),
            "q90_z": float(np.quantile(z[m], 0.90)),
            "q99_z": float(np.quantile(z[m], 0.99)),
        })

window_summary = pd.DataFrame(window_rows)
window_summary.to_csv(DATADIR / "18_window_summary.csv", index=False)

state_summary = pd.DataFrame({
    "state": np.arange(N_STATES),
    "q_edge_low": quantile_edges[:-1],
    "q_edge_high": quantile_edges[1:],
    "count": np.bincount(states, minlength=N_STATES),
})
state_summary.to_csv(DATADIR / "18_state_summary.csv", index=False)

dataset_summary = {
    "notebook_id": NOTEBOOK_ID,
    "x_max": X_MAX,
    "prime_count": int(len(primes)),
    "gap_count_used": int(len(z)),
    "p_min": int(p.min()),
    "p_max": int(p.max()),
    "mean_gap": float(np.mean(gaps)),
    "mean_logp": float(np.mean(logp)),
    "mean_normalized_gap_z": float(np.mean(z)),
    "std_normalized_gap_z": float(np.std(z)),
    "n_states": int(N_STATES),
    "seed": int(SEED),
}
pd.DataFrame([dataset_summary]).to_csv(DATADIR / "18_dataset_summary.csv", index=False)

dataset_summary

## 2. First-order, empirical two-step, and residual operators

The first-order operator is

\[
P_{ij}=\Pr(s_{n+1}=j\mid s_n=i).
\]

The empirical two-step operator is

\[
P^{(2)}_{ik}=\Pr(s_{n+2}=k\mid s_n=i),
\]

and the first-order Markov prediction is

\[
\widehat P^{(2)} = P^2.
\]

Notebook 18 studies the residual matrix

\[
M=P^{(2)}-P^2.
\]

In [ ]:
# ============================================================
# Transition operators and residual matrix
# ============================================================

def row_normalize(M, eps=1e-12):
    M = np.asarray(M, dtype=float)
    row_sum = M.sum(axis=1, keepdims=True)
    return np.divide(M, row_sum + eps, out=np.zeros_like(M), where=row_sum > 0)

def transition_operator(s, n_states=N_STATES):
    C = np.zeros((n_states, n_states), dtype=float)
    for a, b in zip(s[:-1], s[1:]):
        C[a, b] += 1
    return row_normalize(C), C

def two_step_operator(s, n_states=N_STATES):
    C = np.zeros((n_states, n_states), dtype=float)
    for a, c in zip(s[:-2], s[2:]):
        C[a, c] += 1
    return row_normalize(C), C

def frobenius_norm(M):
    return float(np.sqrt(np.sum(np.asarray(M, dtype=float)**2)))

P1, C1 = transition_operator(states)
P2_emp, C2 = two_step_operator(states)
P2_markov = P1 @ P1
M = P2_emp - P2_markov

pd.DataFrame(P1).to_csv(DATADIR / "18_transition_operator_P.csv", index=False)
pd.DataFrame(P2_emp).to_csv(DATADIR / "18_empirical_two_step_operator_P2.csv", index=False)
pd.DataFrame(P2_markov).to_csv(DATADIR / "18_markov_predicted_two_step_operator_P2.csv", index=False)
pd.DataFrame(M).to_csv(DATADIR / "18_two_step_residual_matrix.csv", index=False)

operator_metrics = {
    "markov_l1_error": float(np.mean(np.abs(M))),
    "markov_l2_error": float(np.sqrt(np.mean(M**2))),
    "markov_frobenius_error": frobenius_norm(M),
    "max_abs_residual": float(np.max(np.abs(M))),
    "row_sum_residual_max_abs": float(np.max(np.abs(M.sum(axis=1)))),
}
pd.DataFrame([operator_metrics]).to_csv(DATADIR / "18_operator_metrics.csv", index=False)
operator_metrics

In [ ]:
# ============================================================
# Figure helpers
# ============================================================

figure_paths = []

def savefig(name: str):
    path = FIGDIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    figure_paths.append(path)
    plt.show()
    return path

def heatmap(M, title, xlabel, ylabel, colorbar_label, filename, vmin=None, vmax=None, cmap=None):
    plt.figure(figsize=(8.5, 7))
    im = plt.imshow(M, aspect="auto", origin="lower", vmin=vmin, vmax=vmax, cmap=cmap)
    plt.colorbar(im, label=colorbar_label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    savefig(filename)

In [ ]:
# ============================================================
# Figures: transition and residual operators
# ============================================================

heatmap(P1, "First-order transition operator", "next state", "current state", "probability", "18_transition_operator_P_heatmap.png")
heatmap(P2_emp, "Empirical two-step operator", "state after two steps", "current state", "probability", "18_empirical_two_step_operator_heatmap.png")
heatmap(P2_markov, "Markov-predicted two-step operator", "state after two steps", "current state", "probability", "18_markov_predicted_two_step_operator_heatmap.png")
heatmap(M, "Two-step residual M = empirical - Markov", "state after two steps", "current state", "residual probability", "18_two_step_residual_operator_heatmap.png")

## 3. Spectral decomposition of the residual

We decompose the two-step residual with an SVD:

\[
M = U\Sigma V^\top.
\]

A low-rank correction is

\[
M_k = U_k\Sigma_k V_k^\top,
\]

and a corrected two-step operator is

\[
P^{(2)}_k = P^2 + M_k.
\]

If a small rank captures most of the error, residual memory is structured rather than diffuse.

In [ ]:
# ============================================================
# Spectral decomposition and low-rank corrections
# ============================================================

U, S, Vt = np.linalg.svd(M, full_matrices=False)
energy = S**2
energy_share = energy / max(energy.sum(), 1e-12)
cumulative_energy = np.cumsum(energy_share)

svd_df = pd.DataFrame({
    "mode": np.arange(1, len(S) + 1),
    "singular_value": S,
    "energy_share": energy_share,
    "cumulative_energy": cumulative_energy,
})
svd_df.to_csv(DATADIR / "18_residual_singular_values.csv", index=False)

rank_rows = []
corrected_ops = {}
for k in range(0, TOP_RANK + 1):
    if k == 0:
        Mk = np.zeros_like(M)
    else:
        Mk = (U[:, :k] * S[:k]) @ Vt[:k, :]
    P2k = P2_markov + Mk
    corrected_ops[k] = P2k
    err = P2_emp - P2k
    rank_rows.append({
        "rank_k": k,
        "l1_error": float(np.mean(np.abs(err))),
        "l2_error": float(np.sqrt(np.mean(err**2))),
        "frobenius_error": frobenius_norm(err),
        "captured_energy": float(cumulative_energy[k-1]) if k > 0 else 0.0,
        "max_abs_error": float(np.max(np.abs(err))),
        "min_probability_uncorrected": float(P2k.min()),
        "max_probability_uncorrected": float(P2k.max()),
    })

rank_metrics = pd.DataFrame(rank_rows)
rank_metrics.to_csv(DATADIR / "18_low_rank_prediction_errors.csv", index=False)

# Save representative corrected operators.
for k in [1, 2, 3, 4, 5, 6, 7, 8]:
    pd.DataFrame(corrected_ops[k]).to_csv(DATADIR / f"18_corrected_two_step_operator_rank_{k}.csv", index=False)

svd_df, rank_metrics

In [ ]:
# ============================================================
# Figures: singular values and prediction error by rank
# ============================================================

plt.figure()
plt.plot(svd_df["mode"], svd_df["singular_value"], marker="o", label="singular value")
plt.title("Residual singular-value spectrum")
plt.xlabel("memory mode")
plt.ylabel("singular value")
plt.legend()
savefig("18_residual_singular_value_spectrum.png")

plt.figure()
plt.plot(svd_df["mode"], svd_df["cumulative_energy"], marker="o", label="cumulative energy")
plt.axhline(0.90, linestyle="--", label="90%")
plt.axhline(0.95, linestyle="--", label="95%")
plt.title("Cumulative residual energy by rank")
plt.xlabel("rank k")
plt.ylabel("captured energy")
plt.ylim(0, 1.05)
plt.legend()
savefig("18_cumulative_residual_energy.png")

plt.figure()
plt.plot(rank_metrics["rank_k"], rank_metrics["l1_error"], marker="o", label="L1 error")
plt.plot(rank_metrics["rank_k"], rank_metrics["l2_error"], marker="o", label="L2 error")
plt.title("Predictive error versus memory rank")
plt.xlabel("rank k (0 = Markov P²)")
plt.ylabel("error")
plt.legend()
savefig("18_predictive_error_vs_memory_rank.png")

plt.figure()
plt.bar(["markov_P2"] + [f"rank_{k}" for k in range(1, TOP_RANK + 1)], rank_metrics["frobenius_error"])
plt.xticks(rotation=45, ha="right")
plt.title("Two-step operator prediction error")
plt.xlabel("model")
plt.ylabel("Frobenius error")
savefig("18_two_step_operator_prediction_error.png")

## 4. Residue projections of spectral memory modes

The residual modes live in quantile-state space. To connect them back to arithmetic structure, Notebook 18 projects state-mode weights back onto anchor residues \(p_n\bmod 30\).

In [ ]:
# ============================================================
# Residue projection of singular vectors
# ============================================================

# Map each transition anchor into current state and residue.
state_residue_counts = pd.crosstab(pd.Series(states, name="state"), pd.Series(anchor_mod30, name="residue"))
state_residue_counts = state_residue_counts.reindex(index=np.arange(N_STATES), columns=valid_residues, fill_value=0)
state_residue_probs = state_residue_counts.div(state_residue_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
state_residue_probs.to_csv(DATADIR / "18_state_residue_projection_matrix.csv")

projection_rows = []
for mode_idx in range(min(4, len(S))):
    u_mode = U[:, mode_idx]
    v_mode = Vt[mode_idx, :]
    u_res = state_residue_probs.T.values @ u_mode
    v_res = state_residue_probs.T.values @ v_mode
    for residue, u_val, v_val in zip(valid_residues, u_res, v_res):
        projection_rows.append({
            "mode": int(mode_idx + 1),
            "residue_mod30": int(residue),
            "current_state_vector_U": float(u_val),
            "two_step_state_vector_V": float(v_val),
        })

projection_df = pd.DataFrame(projection_rows)
projection_df.to_csv(DATADIR / "18_residue_projection_by_memory_mode.csv", index=False)

for mode_idx in range(min(4, len(S))):
    sub = projection_df[projection_df["mode"] == mode_idx + 1]
    x = np.arange(len(sub))
    width = 0.38
    plt.figure()
    plt.bar(x - width/2, sub["current_state_vector_U"], width, label="current-state vector U")
    plt.bar(x + width/2, sub["two_step_state_vector_V"], width, label="two-step-state vector V")
    plt.axhline(0, linestyle="--")
    plt.xticks(x, sub["residue_mod30"])
    plt.title(f"Residue projection for memory mode {mode_idx + 1}")
    plt.xlabel("residue mod 30")
    plt.ylabel("singular-vector component")
    plt.legend()
    savefig(f"18_residue_projection_memory_mode_{mode_idx + 1}.png")

projection_df.head()

## 5. Corrected operators and remaining residuals

We inspect low-rank corrected operators and residuals after rank-k correction. This identifies how much structured memory remains after the dominant spectral components are removed.

In [ ]:
# ============================================================
# Corrected operator and remaining residual maps
# ============================================================

for k in [1, 2, 3, 4]:
    Mk = (U[:, :k] * S[:k]) @ Vt[:k, :]
    remaining = M - Mk
    corrected = P2_markov + Mk
    heatmap(corrected, f"Corrected two-step operator P² + M_{k}", "next state", "current state", "probability", f"18_corrected_two_step_operator_rank_{k}.png")
    heatmap(remaining, f"Residual after rank-{k} correction", "next state", "current state", "remaining delta", f"18_remaining_residual_after_rank_{k}.png")

## 6. Windowed low-rank memory diagnostics

Global low-rank structure can be dominated by one scale range. We repeat the low-rank analysis inside logarithmic windows, measuring the rank needed to capture residual energy locally.

In [ ]:
# ============================================================
# Windowed spectral memory diagnostics
# ============================================================

window_rank_rows = []
window_singular_rows = []
window_rank4_remaining = []

for _, row in window_summary.iterrows():
    wi = int(row["window_index"])
    m = window_index == wi
    s_w = states[m]
    if len(s_w) < 30:
        continue
    P1_w, _ = transition_operator(s_w)
    P2_w, _ = two_step_operator(s_w)
    Mw = P2_w - (P1_w @ P1_w)
    Uw, Sw, Vtw = np.linalg.svd(Mw, full_matrices=False)
    ew = Sw**2
    share = ew / max(ew.sum(), 1e-12)
    cum = np.cumsum(share)
    rank90 = int(np.searchsorted(cum, 0.90) + 1)
    rank95 = int(np.searchsorted(cum, 0.95) + 1)

    for mode_idx, sval in enumerate(Sw, start=1):
        window_singular_rows.append({
            "window_index": wi,
            "window_midpoint": float(row["window_midpoint"]),
            "mode": mode_idx,
            "singular_value": float(sval),
            "energy_share": float(share[mode_idx - 1]),
            "cumulative_energy": float(cum[mode_idx - 1]),
        })

    for k in range(0, N_STATES + 1):
        if k == 0:
            Mkw = np.zeros_like(Mw)
        else:
            Mkw = (Uw[:, :k] * Sw[:k]) @ Vtw[:k, :]
        Ew = Mw - Mkw
        window_rank_rows.append({
            "window_index": wi,
            "window_midpoint": float(row["window_midpoint"]),
            "count": int(row["count"]),
            "rank_k": k,
            "l1_remaining": float(np.mean(np.abs(Ew))),
            "l2_remaining": float(np.sqrt(np.mean(Ew**2))),
            "frobenius_remaining": frobenius_norm(Ew),
            "captured_energy": float(cum[k - 1]) if k > 0 else 0.0,
            "rank90": rank90,
            "rank95": rank95,
        })

    k = min(4, len(Sw))
    M4 = (Uw[:, :k] * Sw[:k]) @ Vtw[:k, :]
    window_rank4_remaining.append(np.abs(Mw - M4).reshape(-1))

window_rank_metrics = pd.DataFrame(window_rank_rows)
window_singular_values = pd.DataFrame(window_singular_rows)
window_rank_metrics.to_csv(DATADIR / "18_windowed_low_rank_memory_metrics.csv", index=False)
window_singular_values.to_csv(DATADIR / "18_windowed_singular_values.csv", index=False)

rank0 = window_rank_metrics[window_rank_metrics["rank_k"] == 0]
rank4 = window_rank_metrics[window_rank_metrics["rank_k"] == 4]

plt.figure()
plt.plot(rank0["window_midpoint"], rank0["l2_remaining"], marker="o", label="Markov residual")
plt.plot(rank4["window_midpoint"], rank4["l2_remaining"], marker="o", label="after rank-4 correction")
plt.xscale("log")
plt.title("Windowed L2 residual before and after rank-4 correction")
plt.xlabel("window midpoint x")
plt.ylabel("L2 residual")
plt.legend()
savefig("18_windowed_rank4_residual_reduction.png")

plt.figure()
plt.plot(rank0["window_midpoint"], rank0["rank90"], marker="o", label="rank for 90% energy")
plt.plot(rank0["window_midpoint"], rank0["rank95"], marker="o", label="rank for 95% energy")
plt.xscale("log")
plt.title("Windowed spectral rank requirement")
plt.xlabel("window midpoint x")
plt.ylabel("rank")
plt.legend()
savefig("18_windowed_rank_requirement.png")

if window_rank4_remaining:
    flat = np.stack(window_rank4_remaining, axis=0)
    pair_labels = [f"{i}->{j}" for i in range(N_STATES) for j in range(N_STATES)]
    pd.DataFrame(flat, columns=pair_labels).to_csv(DATADIR / "18_windowed_remaining_rank4_residual_flat.csv", index=False)
    plt.figure(figsize=(12, 6))
    im = plt.imshow(flat, aspect="auto", origin="lower")
    plt.colorbar(im, label="abs remaining delta")
    plt.title("Windowed remaining residual after rank-4 correction")
    plt.xlabel("state-pair transition")
    plt.ylabel("window index")
    plt.xticks(np.arange(0, len(pair_labels), 8), pair_labels[::8], rotation=45, ha="right")
    savefig("18_windowed_remaining_rank4_residual_heatmap.png")

window_rank_metrics.head()

## 7. Shuffle baseline for low-rank spectral memory

To test whether singular-value concentration is just a consequence of finite samples, we compare the real residual spectrum with shuffled sequences that preserve state frequencies while destroying ordering.

In [ ]:
# ============================================================
# Shuffle baseline for spectral residuals
# ============================================================

N_SHUFFLES = 64
shuffle_rows = []
shuffle_singular_values = []

for sid in range(N_SHUFFLES):
    s_shuf = states.copy()
    rng.shuffle(s_shuf)
    P1_s, _ = transition_operator(s_shuf)
    P2_s, _ = two_step_operator(s_shuf)
    Ms = P2_s - (P1_s @ P1_s)
    Ss = np.linalg.svd(Ms, compute_uv=False)
    e = Ss**2
    share = e / max(e.sum(), 1e-12)
    cum = np.cumsum(share)
    shuffle_rows.append({
        "shuffle_id": sid,
        "frobenius_residual": frobenius_norm(Ms),
        "l2_residual": float(np.sqrt(np.mean(Ms**2))),
        "top_singular_value": float(Ss[0]),
        "rank90": int(np.searchsorted(cum, 0.90) + 1),
        "rank95": int(np.searchsorted(cum, 0.95) + 1),
    })
    for mode_idx, sval in enumerate(Ss, start=1):
        shuffle_singular_values.append({
            "shuffle_id": sid,
            "mode": mode_idx,
            "singular_value": float(sval),
            "energy_share": float(share[mode_idx - 1]),
            "cumulative_energy": float(cum[mode_idx - 1]),
        })

shuffle_spectral_summary = pd.DataFrame(shuffle_rows)
shuffle_singular_df = pd.DataFrame(shuffle_singular_values)
shuffle_spectral_summary.to_csv(DATADIR / "18_shuffle_spectral_summary.csv", index=False)
shuffle_singular_df.to_csv(DATADIR / "18_shuffle_singular_values.csv", index=False)

real_rank90 = int(np.searchsorted(cumulative_energy, 0.90) + 1)
real_rank95 = int(np.searchsorted(cumulative_energy, 0.95) + 1)

shuffle_compare = {
    "real_frobenius_residual": frobenius_norm(M),
    "shuffle_mean_frobenius_residual": float(shuffle_spectral_summary["frobenius_residual"].mean()),
    "shuffle_std_frobenius_residual": float(shuffle_spectral_summary["frobenius_residual"].std()),
    "real_top_singular_value": float(S[0]),
    "shuffle_mean_top_singular_value": float(shuffle_spectral_summary["top_singular_value"].mean()),
    "shuffle_std_top_singular_value": float(shuffle_spectral_summary["top_singular_value"].std()),
    "real_rank90": real_rank90,
    "real_rank95": real_rank95,
    "shuffle_mean_rank90": float(shuffle_spectral_summary["rank90"].mean()),
    "shuffle_mean_rank95": float(shuffle_spectral_summary["rank95"].mean()),
}
pd.DataFrame([shuffle_compare]).to_csv(DATADIR / "18_real_vs_shuffle_spectral_comparison.csv", index=False)

plt.figure()
plt.hist(shuffle_spectral_summary["top_singular_value"], bins=18, alpha=0.75, label="shuffle baseline")
plt.axvline(S[0], linestyle="--", linewidth=2, label="real sequence")
plt.title("Top singular value: real versus shuffle baseline")
plt.xlabel("top singular value")
plt.ylabel("frequency")
plt.legend()
savefig("18_real_vs_shuffle_top_singular_value.png")

plt.figure()
plt.hist(shuffle_spectral_summary["frobenius_residual"], bins=18, alpha=0.75, label="shuffle baseline")
plt.axvline(frobenius_norm(M), linestyle="--", linewidth=2, label="real sequence")
plt.title("Residual norm: real versus shuffle baseline")
plt.xlabel("Frobenius residual")
plt.ylabel("frequency")
plt.legend()
savefig("18_real_vs_shuffle_residual_norm.png")

shuffle_compare

## 8. Interpretation numbers

This cell prints compact, paper-friendly numbers for Notebook 18.

In [ ]:
# ============================================================
# Interpretation summary
# ============================================================

rank4_row = rank_metrics.loc[rank_metrics["rank_k"] == 4].iloc[0]
rank1_row = rank_metrics.loc[rank_metrics["rank_k"] == 1].iloc[0]
rank0_row = rank_metrics.loc[rank_metrics["rank_k"] == 0].iloc[0]

interpretation = {
    "markov_l2_error": float(rank0_row["l2_error"]),
    "rank1_l2_error": float(rank1_row["l2_error"]),
    "rank4_l2_error": float(rank4_row["l2_error"]),
    "rank4_error_reduction_fraction": float(1.0 - rank4_row["l2_error"] / max(rank0_row["l2_error"], 1e-12)),
    "top_singular_energy_share": float(energy_share[0]),
    "rank90": int(real_rank90),
    "rank95": int(real_rank95),
    "real_top_singular_value": float(S[0]),
    "shuffle_mean_top_singular_value": float(shuffle_compare["shuffle_mean_top_singular_value"]),
    "top_singular_real_minus_shuffle_zscore": float((S[0] - shuffle_compare["shuffle_mean_top_singular_value"]) / (shuffle_compare["shuffle_std_top_singular_value"] + 1e-12)),
    "real_frobenius_residual": float(shuffle_compare["real_frobenius_residual"]),
    "shuffle_mean_frobenius_residual": float(shuffle_compare["shuffle_mean_frobenius_residual"]),
}

interpretation_df = pd.DataFrame([interpretation])
interpretation_df.to_csv(DATADIR / "18_interpretation_summary.csv", index=False)

for k, v in interpretation.items():
    if isinstance(v, float):
        print(f"{k}: {v:.6g}")
    else:
        print(f"{k}: {v}")

interpretation

## 9. Markdown + LaTeX exports

This locked-template cell writes a short interpretation document and a LaTeX section.

In [ ]:
# ============================================================
# Documentation and LaTeX exports
# ============================================================

figures_md = "\n\n## Figures\n\n"
for idx, fig in enumerate(sorted(FIGDIR.glob("*.png")), start=1):
    title = fig.stem.replace("_", " ").title()
    figures_md += f"### Figure {idx} — {title}\n\n"
    figures_md += f"![Figure {idx}](../figures/{fig.name})\n\n"

summary_lines = [
    f"# {NOTEBOOK_ID}: spectral memory + low-rank operator correction",
    "",
    "## Purpose",
    "",
    "Notebook 18 tests whether the empirical two-step transition residual from Notebook 17 has low-rank spectral structure.",
    "It decomposes M = P2_empirical - P1^2 and measures how much predictive error is removed by rank-k corrections.",
    "",
    "## Key numbers",
    "",
]
for k, v in interpretation.items():
    if isinstance(v, float):
        summary_lines.append(f"- `{k}`: `{v:.6g}`")
    else:
        summary_lines.append(f"- `{k}`: `{v}`")

summary_lines += [
    "",
    "## Interpretation",
    "",
    "A concentrated singular spectrum means the two-step residual is not merely diffuse noise.",
    "A low-rank correction P^2 + M_k estimates how many memory modes are needed to explain the empirical two-step operator.",
    "Residue projections connect dominant spectral modes back to anchor classes modulo 30.",
    "Shuffle comparisons control for finite-sample effects and the one-point state distribution.",
    "",
    figures_md,
]

(DOCDIR / "18_interpretation_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")

tex = rf"""
\section{{Notebook 18: Spectral Memory and Low-Rank Operator Correction}}

Notebook 18 studies whether the two-step transition residual from the normalized
prime-gap state process is low-rank.  Let
\[
z_n = \frac{{p_{{n+1}}-p_n}}{{\log p_n}}
\]
be the normalized prime gap, and let $s_n$ be a quantile-discretized state of
$z_n$.  The first-order transition operator is
\[
P_{{ij}}=\Pr(s_{{n+1}}=j\mid s_n=i),
\]
and the empirical two-step operator is
\[
P^{{(2)}}_{{ik}}=\Pr(s_{{n+2}}=k\mid s_n=i).
\]
The Markov prediction is $\widehat{{P}}^{{(2)}}=P^2$, so the two-step memory
residual is
\[
M=P^{{(2)}}-P^2.
\]
We decompose this residual as
\[
M=U\Sigma V^\top
\]
and define a rank-$k$ correction
\[
P^{{(2)}}_k=P^2+U_k\Sigma_kV_k^\top.
\]

For this run, the Markov $L_2$ error is
\[
{interpretation['markov_l2_error']:.6g},
\]
while the rank-4 corrected $L_2$ error is
\[
{interpretation['rank4_l2_error']:.6g}.
\]
The rank-4 correction reduces error by
\[
{interpretation['rank4_error_reduction_fraction']:.6g}.
\]
The leading singular mode accounts for energy share
\[
{interpretation['top_singular_energy_share']:.6g},
\]
and the rank needed to capture 90\% of residual energy is
\[
{interpretation['rank90']}.
\]
A shuffle baseline gives mean top singular value
\[
{interpretation['shuffle_mean_top_singular_value']:.6g},
\]
compared with the real top singular value
\[
{interpretation['real_top_singular_value']:.6g}.
\]
These diagnostics test whether higher-order transition memory is structured as a
small number of spectral modes rather than spread uniformly across transitions.
"""
(TEXDIR / "18_spectral_memory_low_rank_operator.tex").write_text(tex, encoding="utf-8")

print("Wrote docs and TeX:")
print(DOCDIR / "18_interpretation_summary.md")
print(TEXDIR / "18_spectral_memory_low_rank_operator.tex")

## 10. Manifest and optional Colab download

The final cell writes a manifest and creates the standard root-level export zip.

Locked-template standard:

```python
# from google.colab import files
# files.download("18_spectral_memory_low_rank_operator_export.zip")
```

In [ ]:
# ============================================================
# Manifest + locked-template export zip
# ============================================================

manifest_rows = []
for subdir in [FIGDIR, DATADIR, DOCDIR, TEXDIR]:
    for path in sorted(subdir.glob("*")):
        if path.is_file():
            manifest_rows.append({
                "notebook_id": NOTEBOOK_ID,
                "relative_path": str(path),
                "folder": path.parent.name,
                "filename": path.name,
                "size_bytes": path.stat().st_size,
            })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(DATADIR / "18_outputs_manifest.csv", index=False)

EXPORT_ZIP = Path(f"{NOTEBOOK_ID}_export.zip")
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTDIR.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=str(path))

print("Export zip created:", EXPORT_ZIP)
print("Files in manifest:", len(manifest))
print("Zip size bytes:", EXPORT_ZIP.stat().st_size)

# Optional: download outputs bundle (template standard)
# from google.colab import files
# files.download("18_spectral_memory_low_rank_operator_export.zip")